#### 02 - Disambiguation - Evaluation

In this notebook, we will run CG3 disambiguation function and compare with gold result

In [1]:
# !pip install sequence_align==0.3.0

In [2]:
# Import cg3 and fst processing library 
import sys 
sys.path.append("../src")
import cg3_process as cg3 

In [3]:
from sequence_align.pairwise import needleman_wunsch

In [4]:
GRAMMAR_FILE = "../data/CG3_rules/Ojibwe_disambiguation.cg3"
FST_BINARY_FILE = "../data/fst/ojibwe.att"
ORGINAL_READINGS_FILE = "../data/evaluation/sample/eval_original_fst_readings.txt"
DISAMBIGUATED_FILE = "../data/evaluation/sample/eval_disambiguated_SYS.txt"
GOLD_DISAMBIGUATED_FILE = "../data/evaluation/sample/eval_disambiguated_GOLD.txt"
GAP_TOKEN = "_"

In [5]:
def read_file(filename: str) -> list[str]:
    """
    Read a file and return its contents as a list of lines.

    Parameters
    ----------
    filename : str
        Path to the file to be read.

    Returns
    -------
    list[str]
        A list of lines from the file.
    """
    try:
        with open(filename, 'r', encoding="utf-8") as file:
            return file.readlines()
    except Exception as e:
        print(f"Error reading file {filename}: {e}")
        return []

In [6]:
original_readings = read_file(ORGINAL_READINGS_FILE)
print(f"Number of original readings: {len(original_readings)}")

# Quick check the original readings
for i in range(10):
    print(f"Original reading {i}: {original_readings[i].strip()}")

Number of original readings: 128
Original reading 0: "<onzaam>"
Original reading 1: "onzaam" ADVQnt
Original reading 2: "<niibowa>"
Original reading 3: "niibowa" ADVQnt
Original reading 4: "<oginigawinaan>"
Original reading 5: "ginigawin" VTA Ind Pos Neu 3SgProxSubj 3SgObvObj
Original reading 6: "ginigawin" VTA Ind Pos Neu 3SgProxSubj 3PlObvObj
Original reading 7: "ginigawinan" VTI Ind Pos Neu 3SgProxSubj 0SgObj
Original reading 8: "<'i>"
Original reading 9: 


In [7]:
# Combine into a single string
original_readings_str = ''.join(original_readings)

print(original_readings_str[:500])  # Print the first characters for a quick check

"<onzaam>"
	"onzaam" ADVQnt
"<niibowa>"
	"niibowa" ADVQnt
"<oginigawinaan>"
	"ginigawin" VTA Ind Pos Neu 3SgProxSubj 3SgObvObj
	"ginigawin" VTA Ind Pos Neu 3SgProxSubj 3PlObvObj
	"ginigawinan" VTI Ind Pos Neu 3SgProxSubj 0SgObj
"<'i>"

"<wiisagad>"
	"wiisagad" NI Sg
"<imaa>"
	"imaa" ADVLoc
"<g...>"

"<iwe>"
	"iwe" PRONDem NI Sg
"<gii-naboobiiked>"
	"naboobiike" PVTense/gii VAI Cnj Pos Neu 3SgProxSubj
"<.>"

"<ningii-wanitoon>"
	"wanitoon" PVTense/gii VTI Ind Pos Neu 1SgSubj 0SgObj
"<niwaazakoneb


In [8]:
# Run the disambiguation process
disambiguated_readings = cg3.cg3_process_text(input_text=original_readings_str,
                                              cg3_grammar_filepath=GRAMMAR_FILE)

# Print the first 500 characters of the disambiguated readings
print(disambiguated_readings[:500])  

"<onzaam>"
	"onzaam" ADVQnt
"<niibowa>"
	"niibowa" ADVQnt
"<oginigawinaan>"
	"ginigawinan" VTI Ind Pos Neu 3SgProxSubj 0SgObj
"<'i>"
"<wiisagad>"
	"wiisagad" NI Sg
"<imaa>"
	"imaa" ADVLoc
"<g...>"
"<iwe>"
	"iwe" PRONDem NI Sg
"<gii-naboobiiked>"
	"naboobiike" PVTense/gii VAI Cnj Pos Neu 3SgProxSubj
"<.>"
"<ningii-wanitoon>"
	"wanitoon" PVTense/gii VTI Ind Pos Neu 1SgSubj 0SgObj
"<niwaazakonebijigan>"
	"waazakonebijigan" NI Sg 1SgPoss
"<wii-aabajitooyaan>"
	"aabajitoon" PVTense/wii VTI Cnj Pos Ne


In [9]:
def write_to_file(filename: str, lines: list[str]) -> None:
    """
    Write a list of lines to a file.

    Parameters
    ----------
    filename : str
        Path to the file to write to.
    lines : list[str]
        The list of lines to be written to the file.
    """
    try:
        with open(filename, 'w', encoding="utf-8") as file:
            for line in lines:
                file.write(line + "\n")
        print(f"Output written to {filename}")
    except Exception as e:
        print(f"Error writing to file {filename}: {e}")

In [10]:
# Write the output to a file
output_lines = disambiguated_readings.splitlines()
write_to_file(DISAMBIGUATED_FILE, output_lines)

Output written to ../data/evaluation/sample/eval_disambiguated_SYS.txt


#### Evaluation

Compare the system disambiguated with gold output, word by word

In [11]:
def compare_readings(sys_readings: list[str], gold_readings: list[str]) -> tuple:
    """
    Compare system output readings with the gold standard.

    Parameters
    ----------
    sys_readings : list[str]
        The system output readings.
    gold_readings : list[str]
        The gold standard readings.

    Returns
    -------
    tuple
        A tuple containing the precision and recall scores.
    """
    if len(sys_readings) == 0 and len(gold_readings) == 0:
        print("Warning: Both system and gold readings are empty!")
        return (1, 1)
    elif len(sys_readings) == 0 or len(gold_readings) == 0:
        print("Warning: One of the readings is empty!")
        return (0, 0)

    # Remove empty lines
    sys_lines = [line for line in sys_readings if line.strip()]
    gold_lines = [line for line in gold_readings if line.strip()]
    
    
    # Check if the word-form is matching
    sys_lemma = sys_lines[0].strip('"<>\t ')
    gold_lemma = gold_lines[0].strip('"<>\t ')
    
    if sys_lemma != gold_lemma:
        print(f"Warning: Word-form do not match! System: {sys_lemma}, Gold: {gold_lemma}")
        return (0, 0)
    
    correct_count = 0 
    for line in sys_lines[1:]:
        if line in gold_lines:
            correct_count += 1
    
    sys_count = len(sys_lines) - 1  # Exclude the word-form line
    gold_count = len(gold_lines) - 1  # Exclude the word-form line
    
    
    precision = correct_count / sys_count if sys_count > 0 else 1
    recall = correct_count / gold_count if gold_count > 0 else 1  
    
    return (precision, recall) 

# Test a simple case 
sys_readings = ['"<oginigawinaan>"',
        '   "ginigawin" VTA Ind Pos Neu 3SgProxSubj 3SgObvObj',
        '   "ginigawinan" VTI Ind Pos Neu 3SgProxSubj 0SgObj'
        ]

gold_readings = ['"<oginigawinaan>"',
        '   "ginigawinan" VTI Ind Pos Neu 3SgProxSubj 0SgObj'
        ]
        

precision, recall = compare_readings(sys_readings, gold_readings)
assert precision == 1/2
assert recall == 1/1
print(f"Precision: {precision}, Recall: {recall}")

assert compare_readings([], []) == (1.0, 1.0)
assert compare_readings(['<a>'], ['<a>']) == (1.0, 1.0)
assert compare_readings(['<a>'], ['<b>']) == (0.0, 0.0)

Precision: 0.5, Recall: 1.0


In [12]:
# Load the system and gold standard readings
sys_readings = read_file(DISAMBIGUATED_FILE)
gold_readings = read_file(GOLD_DISAMBIGUATED_FILE)

print(f"Number of system readings: {len(sys_readings)}")
print(f"Number of gold readings: {len(gold_readings)}")

print(sys_readings[:5])  # Print the first 5 lines for a quick check
print(gold_readings[:5])  # Print the first 5 lines for a quick check

Number of system readings: 102
Number of gold readings: 108
['"<onzaam>"\n', '\t"onzaam" ADVQnt\n', '"<niibowa>"\n', '\t"niibowa" ADVQnt\n', '"<oginigawinaan>"\n']
['"<onzaam>"\n', '        "onzaam" ADVQnt\n', '"<niibowa>"\n', '        "niibowa" ADVQnt\n', '"<oginigawinaan>"\n']


In [13]:
WORD_FORM_START_TAG = '"<'
WORD_FORM_END_TAG = '>"'

In [14]:
# Groups the readings by word-form 
def group_readings_by_word_form(readings_lines: list[str]) -> list[str]:
    """
    Group readings by word form.

    Parameters
    ----------
    readings_lines : list[str]
        A list of readings to be grouped.

    Returns
    -------
    list[str]
        A list of grouped readings by word form.
    """
    grouped_readings = []
    current_word_form = ""
    
    current_reading_group = []
    
    for line in readings_lines:
        line = line.strip()
        # Check if the line starts with a word-form tag
        if line.startswith(WORD_FORM_START_TAG) and line.endswith(WORD_FORM_END_TAG):
            # Append the current reading group if it exists
            if len(current_reading_group) > 0:
                grouped_readings.append(current_reading_group)
                current_reading_group = []
                
            # Start a new group with the current word-form
            current_word_form = line
            current_reading_group.append(current_word_form)
        else:
            current_reading_group.append(line)
    
    return grouped_readings

sys_readings_group = group_readings_by_word_form(sys_readings)
gold_readings_group = group_readings_by_word_form(gold_readings)

# Print the first group for a quick check
print(f"System readings group: {sys_readings_group[0]}")
print(f"Gold readings group: {gold_readings_group[0]}")


System readings group: ['"<onzaam>"', '"onzaam" ADVQnt']
Gold readings group: ['"<onzaam>"', '"onzaam" ADVQnt']


In [15]:
def align_readings(sys_readings_group: list[str], gold_readings_group: list[str]) -> list[tuple]:
    """
    Align system and gold readings by word form using the Needleman-Wunsch algorithm,
    to avoid the problems of extra or missing punctuations 

    Parameters
    ----------
    sys_readings_group : list[str]
        The system readings grouped by word form.
    gold_readings_group : list[str]
        The gold standard readings grouped by word form.

    Returns
    -------
    list[tuple]
        A list of tuples containing the aligned system and gold readings.
    """
    aligned_sys = []
    aligned_gold = []
    
    
    # Create a sys and gold list of word-forms 
    sys_word_forms = [group[0] for group in sys_readings_group]
    gold_word_forms = [group[0] for group in gold_readings_group]

    # Align the readings using Needleman-Wunsch distance
    aligned_word_form_sys, aligned_word_form_gold = needleman_wunsch(sys_word_forms, gold_word_forms, gap=GAP_TOKEN)

    # Iterate through the aligned readings and create the final grouped readings
    j = 0 
    for sys_word_form in aligned_word_form_sys:
        if sys_word_form != GAP_TOKEN:
            aligned_sys.append(sys_readings_group[j])
            j += 1
        else:
            aligned_sys.append([GAP_TOKEN])

    j = 0 
    for gold_word_from in aligned_word_form_gold:
        if gold_word_from != GAP_TOKEN:
            aligned_gold.append(gold_readings_group[j])
            j += 1
        else:
            aligned_gold.append([GAP_TOKEN])
            
    return aligned_sys, aligned_gold

# Simple test case
sys_list = [
    ["a", "a1", "a2"],
    ["b", "b1", "b2"],
    ["."],
    ["c", "c1", "c2"]
]

gold_list = [
    ["a", "a1", "a2", "a3"],
    ["b", "b1", "b2"],
    ["c", "c1", "c2"]  # missing punctuation in gold 
]

aligned_sys, aligned_gold = align_readings(sys_list, gold_list)
assert aligned_sys[2][0] == "."
assert aligned_gold[2][0] == GAP_TOKEN

In [16]:
# Align the system and gold readings
aligned_sys, aligned_gold = align_readings(sys_readings_group, gold_readings_group)
# Print the first 3 aligned readings for a quick check  
print(f"Aligned system readings: {aligned_sys[:3]}")
print(f"Aligned gold readings: {aligned_gold[:3]}")

Aligned system readings: [['"<onzaam>"', '"onzaam" ADVQnt'], ['"<niibowa>"', '"niibowa" ADVQnt'], ['"<oginigawinaan>"', '"ginigawinan" VTI Ind Pos Neu 3SgProxSubj 0SgObj']]
Aligned gold readings: [['"<onzaam>"', '"onzaam" ADVQnt'], ['"<niibowa>"', '"niibowa" ADVQnt'], ['"<oginigawinaan>"', '"ginigawinan" VTI Ind Pos Neu 3SgProxSubj 0SgObj']]


Now we can run through both system and gold groups to get the precision and recall scores for each word-form

In [17]:
def compare_readings_groups(sys_readings_group: list[str], gold_readings_group: list[str]) -> tuple:
    """
    Compare system output readings with the gold standard.

    Parameters
    ----------
    sys_readings_group : list[str]
        The system output readings grouped by word form.
    gold_readings_group : list[str]
        The gold standard readings grouped by word form.

    Returns
    -------
    tuple
        A tuple containing the average precision and recall scores.
    """

    avg_precision = 0
    avg_recall = 0
    
    precision_scores = []
    recall_scores = []
    
    print(f"Number of system readings groups: {len(sys_readings_group)}")
    print(f"Number of gold readings groups: {len(gold_readings_group)}")
    
    
    for i in range(len(sys_readings_group)):
        sys_readings = sys_readings_group[i]
        gold_readings = gold_readings_group[i]
        
        # Check if the word-form is matching
        sys_word_form = sys_readings[0].strip('"<>\t ')
        gold_word_form = gold_readings[0].strip('"<>\t ')
        
        if sys_word_form != gold_word_form:
            print(f"Warning: Word-form do not match! System: <{sys_word_form}>, Gold: <{gold_word_form}>. Skipping...")
            continue

        
        precision, recall = compare_readings(sys_readings, gold_readings)
        print(f"Precision: {precision:.2f}, Recall: {recall:.2f} for word-form <{sys_word_form}>")
        
        precision_scores.append(precision)
        recall_scores.append(recall)
    
    # Calculate average precision and recall
    avg_precision = sum(precision_scores) / len(precision_scores) if precision_scores else 0
    avg_recall = sum(recall_scores) / len(recall_scores) if recall_scores else 0
    
    return (avg_precision, avg_recall)
    


In [18]:
# avg_precision, avg_recall = compare_readings_groups(sys_readings_group, gold_readings_group)
avg_precision, avg_recall = compare_readings_groups(aligned_sys, aligned_gold)


Number of system readings groups: 54
Number of gold readings groups: 54
Precision: 1.00, Recall: 1.00 for word-form <onzaam>
Precision: 1.00, Recall: 1.00 for word-form <niibowa>
Precision: 1.00, Recall: 1.00 for word-form <oginigawinaan>
Precision: 1.00, Recall: 1.00 for word-form <'i>
Precision: 1.00, Recall: 1.00 for word-form <wiisagad>
Precision: 1.00, Recall: 1.00 for word-form <imaa>
Precision: 1.00, Recall: 1.00 for word-form <g...>
Precision: 1.00, Recall: 1.00 for word-form <iwe>
Precision: 1.00, Recall: 1.00 for word-form <gii-naboobiiked>
Precision: 1.00, Recall: 1.00 for word-form <.>
Precision: 1.00, Recall: 1.00 for word-form <ningii-wanitoon>
Precision: 1.00, Recall: 1.00 for word-form <niwaazakonebijigan>
Precision: 1.00, Recall: 1.00 for word-form <wii-aabajitooyaan>
Precision: 1.00, Recall: 1.00 for word-form <wii-waaswaayaan>
Precision: 1.00, Recall: 1.00 for word-form <.>
Precision: 1.00, Recall: 1.00 for word-form <awiya>
Precision: 1.00, Recall: 1.00 for word-for

In [19]:
print(f"Evaluation Average Precision: {avg_precision:.2f}, Average Recall: {avg_recall:.2f}")
f1_score = 2 * (avg_precision * avg_recall) / (avg_precision + avg_recall) if (avg_precision + avg_recall) > 0 else 0
print(f"F1 Score: {f1_score:.2f}")

Evaluation Average Precision: 0.98, Average Recall: 0.95
F1 Score: 0.97
